
# 60-Minute Mock Interview - Pandas Data Wrangling

**Jane Street Data Engineer practice**

This notebook is a timed diagnostic, not a prediction of the interview. It uses the existing `pandas_workout/datasets` CSVs and the same conventions as the workbook: pure functions, explicit dtypes, validated joins, vectorized operations, and readable method chains.



## Rules

1. Start a 60-minute timer before reading Task 1.
2. Run the setup cell and the fixture cell first.
3. Attempt tasks in order. Run each self-check before moving on.
4. Do not open the collapsible solutions until the timer ends.
5. If a task takes more than 10 minutes, write down the blocker and move on.
6. Do not mutate `BASE`, `CLEAN`, `ORDERS`, `CUSTOMERS`, `PRODUCTS`, or `PRICES`.
7. Score yourself at the end: correctness, purity, vectorization, readability, and method chaining.

Time budget: 8 + 8 + 7 + 9 + 9 + 7 = 48 minutes for the six core tasks, 5 minutes for the bonus, and 7 minutes for review.


In [1]:

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

pd.options.mode.copy_on_write = True
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

DATA_DIR = Path("datasets")
if not DATA_DIR.exists():
    DATA_DIR = Path("pandas_workout") / "datasets"

CUSTOMERS = pd.read_csv(DATA_DIR / "customers.csv", dtype={"customer_id": "string"})
PRODUCTS = pd.read_csv(DATA_DIR / "products.csv", dtype={"product_id": "string"})
PRICES = pd.read_csv(DATA_DIR / "product_prices.csv", dtype={"product_id": "string"})
ORDERS = pd.read_csv(
    DATA_DIR / "orders.csv",
    dtype={"order_id": "string", "customer_id": "string", "product_id": "string"},
    na_values=["", "N/A", "NA", "?"],
)

print("customers", CUSTOMERS.shape)
print("products", PRODUCTS.shape)
print("prices", PRICES.shape)
print("orders", ORDERS.shape)


customers (215, 7)
products (30, 6)
prices (62, 3)
orders (1770, 10)



## Reference fixture for Tasks 2-6

Run this cell once. It builds a known-good `CLEAN` orders frame and `BASE` fact table so that a mistake in Task 1 does not cascade.

Do not copy this fixture for Task 1. Task 1 is graded separately. Do not edit this cell.


In [2]:

def parse_currency(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace("$", "", regex=False)
        .str.replace("USD", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce",
    ).astype("Float64")


def clean_country(series):
    mapping = {
        "united states": "United States", "usa": "United States",
        "us": "United States", "u.s.": "United States",
        "united kingdom": "United Kingdom", "uk": "United Kingdom",
        "u.k.": "United Kingdom", "great britain": "United Kingdom",
        "canada": "Canada", "ca": "Canada", "can": "Canada",
        "germany": "Germany", "de": "Germany", "deutschland": "Germany",
        "france": "France", "fr": "France",
    }
    return (
        series.astype("string").str.strip().str.lower()
        .map(mapping).astype("string").fillna("Unknown")
    )


def clean_segment(series):
    mapping = {
        "consumer": "Consumer", "smb": "SMB", "small business": "SMB",
        "enterprise": "Enterprise", "ent": "Enterprise",
    }
    return (
        series.astype("string").str.strip().str.lower()
        .map(mapping).astype("string").fillna("Unknown")
    )


def clean_category(series):
    mapping = {
        "electronics": "Electronics", "electronic": "Electronics",
        "home office": "Home Office", "home & office": "Home Office",
        "accessories": "Accessories", "accessory": "Accessories",
        "furniture": "Furniture",
    }
    return (
        series.astype("string").str.strip().str.lower()
        .map(mapping).astype("string")
    )


def make_fixture():
    customers = CUSTOMERS.assign(
        country=clean_country(CUSTOMERS["country"]),
        segment=clean_segment(CUSTOMERS["segment"]),
    )
    products = PRODUCTS.assign(
        category=clean_category(PRODUCTS["category"]),
        unit_cost=pd.to_numeric(PRODUCTS["unit_cost"], errors="coerce"),
    )
    updated = pd.to_datetime(ORDERS["updated_at"], format="mixed", errors="coerce")
    orders = (
        ORDERS.assign(_updated=updated)
        .sort_values(["order_id", "_updated"], na_position="first")
        .drop_duplicates("order_id", keep="last")
        .drop(columns="_updated")
    )
    order_date = pd.to_datetime(orders["order_date"], format="mixed", errors="coerce")
    unit_price = parse_currency(orders["unit_price"])
    status = (
        orders["status"].astype("string").str.strip().str.lower()
        .replace({"complete": "completed", "canceled": "cancelled"})
    )
    channel = (
        orders["channel"].astype("string").str.strip().str.lower()
        .replace({"online": "web", "app": "mobile", "retail": "store"})
    )
    clean = orders.assign(
        order_date=order_date,
        updated_at=pd.to_datetime(orders["updated_at"], format="mixed", errors="coerce"),
        unit_price=unit_price,
        status=status,
        channel=channel,
    ).reset_index(drop=True)
    mask = (
        clean["order_date"].notna()
        & clean["customer_id"].isin(customers["customer_id"])
        & clean["product_id"].isin(products["product_id"])
        & clean["quantity"].gt(0)
        & clean["unit_price"].notna()
        & clean["discount"].between(0, 0.5)
    )
    base = (
        clean.loc[mask]
        .merge(
            customers[["customer_id", "country", "segment"]],
            on="customer_id",
            how="left",
            validate="many_to_one",
        )
        .merge(
            products[["product_id", "category", "unit_cost"]],
            on="product_id",
            how="left",
            validate="many_to_one",
        )
        .assign(
            revenue=lambda frame: (
                frame["quantity"] * frame["unit_price"] * (1 - frame["discount"])
            ).round(2)
        )
        .reset_index(drop=True)
    )
    return clean, base


CLEAN, BASE = make_fixture()

assert CLEAN.shape == (1700, 10)
assert BASE.shape[0] == 1480
assert BASE["customer_id"].nunique() == 200
assert np.isclose(BASE["revenue"].sum(), 1299800.79)

print("CLEAN", CLEAN.shape)
print("BASE", BASE.shape)
print("BASE revenue", round(float(BASE["revenue"].sum()), 2))


CLEAN (1700, 10)
BASE (1480, 15)
BASE revenue 1299800.79



## Timer starts now

### Scoring

| Area | Points |
|---|---:|
| Task 1: clean orders | 0-2 |
| Task 2: customer metrics | 0-2 |
| Task 3: join audit | 0-2 |
| Task 4: daily report | 0-2 |
| Task 5: reshape report | 0-2 |
| Task 6: as-of prices | 0-2 |
| Bonus A | 0-2 |
| Bonus B | 0-2 |
| Method chaining and purity | 0-2 |

2 = correct, pure, vectorized, readable. 1 = passes but fragile or imperative. 0 = wrong, missing, or mutates inputs.

12-14 including method chaining: ready. 9-11: close. Below 9: focus on fundamentals.



## Task 1 - Clean orders (8 minutes)

Write `clean_orders(path)` that:

- reads the orders CSV with `order_id`, `customer_id`, and `product_id` as `"string"`;
- treats `""`, `"N/A"`, `"NA"`, and `"?"` as missing;
- parses `order_date` and `updated_at` with `format="mixed"`, `errors="coerce"`;
- parses `unit_price` to `Float64`, handling `$`, `USD`, commas, and whitespace;
- canonicalizes `status` as `"string"`: `complete` -> `completed`, `canceled` -> `cancelled`;
- canonicalizes `channel` as `"string"`: `online` -> `web`, `app` -> `mobile`, `retail` -> `store`;
- deduplicates by `order_id`, keeping the row with the latest `updated_at` (a missing timestamp loses);
- resets the index;
- keeps all 10 columns and all 1,700 deduplicated rows. Do not filter invalid business rows yet.

Use `.pipe` with at least three named helper functions. Do not mutate the input file or any global frame.

**Hint:** `sort_values(["order_id", "_updated"], na_position="first").drop_duplicates("order_id", keep="last")` lets a valid timestamp beat a missing one.


In [10]:

def clean_orders(path):
    df = pd.read_csv(path, 
        dtype = {"order_id" : "string", "customer_id": "string", "product_id": "string"},
        na_values = ["", "N/A", "NA", "?"]
    )

    order_date = pd.to_datetime(df.order_date, errors = "coerce", format = "mixed")
    updated_at = pd.to_datetime(df.updated_at, errors = "coerce", format = "mixed")

    unit_price = df.unit_price.str.strip().str.replace(",", "").str.replace("$", "").str.replace("USD", "").astype("Float64")

    status = df.status.replace({"complete" : "completed", "canceled": "cancelled"})
    channel = df.channel.replace({"online" : "web", "app": "mobile", "retail": "store"})

    df = df.assign(
        order_date = order_date,
        updated_at = updated_at,
        unit_price = unit_price,
        status = status,
        channel = channel
    )

    df = df.sort_values(by = ["order_id", "updated_at"], na_position="last").drop_duplicates(subset = "updated_at")

    df = df.reset_index()

    return df


co = clean_orders(DATA_DIR / "orders.csv")

co

,index,order_id,customer_id,order_date,product_id,quantity,unit_price,discount,status,channel,updated_at
0,524,O000001,C0139,2023-09-10,P005,3.0,381.36,0.15,COMPLETE,web,2023-09-15
1,1048,O000002,C0049,2022-02-23,P029,3.0,<NA>,0.05,pending,Mobile,2022-02-27
2,25,O000003,C0166,2023-01-25,P017,2.0,78.25,0.05,cancelled,store,2023-01-29
3,627,O000004,C0124,2022-06-18,P005,NaN,366.17,0.00,PENDING,mobile,2022-06-18
4,1666,O000005,C0025,2024-10-19,P023,1.0,73.3,0.10,Returned,Web,2024-10-23
...,...,...,...,...,...,...,...,...,...,...,...
881,1545,O001676,C0123,2024-07-31,P027,1.0,245.87,0.30,Pending,store,2024-08-02
882,511,O001679,C0123,2023-06-20,P022,1.0,55.91,0.20,cancelled,mobile,2023-06-20
883,1642,O001686,C0054,2024-02-11,P016,1.0,436.61,0.00,completed,APP,2024-02-11
884,1061,O001688,C0077,2022-08-12,P011,1.0,95.78,0.25,pending,mobile,2022-08-17


In [11]:

co = clean_orders(DATA_DIR / "orders.csv")

assert isinstance(co, pd.DataFrame)
assert co.shape == (1700, 10)
assert set(co.columns) == set(ORDERS.columns)
assert co["order_id"].is_unique
assert str(co["order_id"].dtype) == "string"
assert str(co["customer_id"].dtype) == "string"
assert str(co["product_id"].dtype) == "string"
assert str(co["order_date"].dtype).startswith("datetime64")
assert str(co["updated_at"].dtype).startswith("datetime64")
assert str(co["unit_price"].dtype) == "Float64"
assert str(co["status"].dtype) == "string"
assert str(co["channel"].dtype) == "string"
assert co["order_date"].isna().sum() == 51
assert co["unit_price"].isna().sum() == 20
assert co["status"].value_counts().to_dict() == {
    "completed": 1239,
    "cancelled": 171,
    "pending": 156,
    "returned": 134,
}
assert co["channel"].value_counts().to_dict() == {
    "web": 661,
    "mobile": 620,
    "store": 419,
}
print("Task 1 passed")


AssertionError: 

<details>
<summary>Show reference solution</summary>

```python
def read_orders(path):
    return pd.read_csv(
        path,
        dtype={
            "order_id": "string",
            "customer_id": "string",
            "product_id": "string",
        },
        na_values=["", "N/A", "NA", "?"],
    )


def dedupe_latest(df):
    updated = pd.to_datetime(df["updated_at"], format="mixed", errors="coerce")
    return (
        df.assign(_updated=updated)
        .sort_values(["order_id", "_updated"], na_position="first")
        .drop_duplicates("order_id", keep="last")
        .drop(columns="_updated")
    )


def parse_dates(df):
    return df.assign(
        order_date=pd.to_datetime(df["order_date"], format="mixed", errors="coerce"),
        updated_at=pd.to_datetime(df["updated_at"], format="mixed", errors="coerce"),
    )


def parse_prices(df):
    return df.assign(unit_price=parse_currency(df["unit_price"]))


def normalize_labels(df):
    return df.assign(
        status=(
            df["status"].astype("string").str.strip().str.lower()
            .replace({"complete": "completed", "canceled": "cancelled"})
        ),
        channel=(
            df["channel"].astype("string").str.strip().str.lower()
            .replace({"online": "web", "app": "mobile", "retail": "store"})
        ),
    )


def clean_orders(path):
    return (
        read_orders(path)
        .pipe(dedupe_latest)
        .pipe(parse_dates)
        .pipe(parse_prices)
        .pipe(normalize_labels)
        .reset_index(drop=True)
    )
```

</details>


## Task 2 - Customer metrics (8 minutes)

Use the provided `BASE` fact table. Write `customer_metrics(base)` returning one row per customer, indexed by `customer_id`, with columns in this order:

`country`, `segment`, `n_orders`, `units`, `revenue`, `avg_order_value`, `first_order`, `last_order`, `recency_days`

- `n_orders`: distinct order IDs;
- `units`: sum of quantity;
- `revenue`: sum of revenue;
- `avg_order_value`: mean revenue per order line;
- `first_order` / `last_order`: min/max `order_date`;
- `recency_days`: days from the latest order date in `base` to the customer's last order;
- sort by `customer_id`; do not mutate `base`.

**Hint:** named aggregation first, then `.assign` for `recency_days`.


In [18]:

def customer_metrics(base):
    df = (base
        .groupby("customer_id")
        .agg(
            country = ("country", "first"),
            segment = ("segment", "first"),
            n_orders = ("order_id", "nunique"),
            units =  ("quantity", "sum"),
            avg_order_value = ("unit_price", "mean"),
            first_order = ("order_date", "min"),
            last_order = ("order_date", "max"),
        )
    )

    last_seen_day = max(base.order_date)

    df = df.assign(
        recency_days = lambda x : last_seen_day - x.last_order
    )

    df = df.sort_values(by = "customer_id")

    return df

before = BASE.copy(deep=True)
cm = customer_metrics(BASE)

cm

,country,segment,n_orders,units,avg_order_value,first_order,last_order,recency_days
customer_id,,,,,,,,
C0001,France,Enterprise,2,4.0,480.82,2022-09-18,2023-04-12,629 days
C0002,Germany,SMB,10,217.0,338.799,2022-05-29,2024-11-02,59 days
C0003,Canada,Enterprise,12,35.0,197.576667,2022-06-20,2024-07-25,159 days
C0004,France,Enterprise,9,17.0,160.106667,2022-01-06,2024-12-26,5 days
C0005,United States,SMB,7,20.0,263.012857,2022-07-15,2024-08-01,152 days
...,...,...,...,...,...,...,...,...
C0196,Germany,SMB,3,6.0,200.433333,2022-01-28,2024-07-07,177 days
C0197,Germany,SMB,4,6.0,89.0625,2022-04-06,2023-11-05,422 days
C0198,France,SMB,10,20.0,141.402,2022-05-12,2023-11-20,407 days


In [19]:

before = BASE.copy(deep=True)
cm = customer_metrics(BASE)

assert isinstance(cm, pd.DataFrame)
assert cm.index.name == "customer_id"
assert cm.index.is_unique
assert list(cm.columns) == [
    "country", "segment", "n_orders", "units", "revenue",
    "avg_order_value", "first_order", "last_order", "recency_days",
]
assert len(cm) == 200
assert np.isclose(cm["revenue"].sum(), 1299800.79)
assert int(cm["n_orders"].sum()) == 1480
assert int(cm["units"].sum()) == 4973
assert cm["first_order"].min() == pd.Timestamp("2022-01-01")
assert cm["last_order"].max() == pd.Timestamp("2024-12-31")
assert int(cm["recency_days"].max()) == 629
assert set(cm["country"]) == {
    "Canada", "France", "Germany", "United Kingdom", "United States", "Unknown",
}
assert set(cm["segment"]) == {"Consumer", "SMB", "Enterprise", "Unknown"}
assert np.allclose(cm["avg_order_value"], cm["revenue"] / cm["n_orders"])
assert str(cm["first_order"].dtype).startswith("datetime64")
assert str(cm["last_order"].dtype).startswith("datetime64")
pd.testing.assert_frame_equal(BASE, before)
print("Task 2 passed")


AssertionError: 

<details>
<summary>Show reference solution</summary>

```python
def customer_metrics(base):
    out = (
        base.groupby("customer_id", observed=True)
        .agg(
            country=("country", "first"),
            segment=("segment", "first"),
            n_orders=("order_id", "nunique"),
            units=("quantity", "sum"),
            revenue=("revenue", "sum"),
            avg_order_value=("revenue", "mean"),
            first_order=("order_date", "min"),
            last_order=("order_date", "max"),
        )
        .sort_index()
    )
    return out.assign(
        recency_days=(base["order_date"].max() - out["last_order"]).dt.days
    )
```

</details>


## Task 3 - Join audit (7 minutes)

Write `audit_joins(orders, customers, products)` using the raw frames `ORDERS`, `CUSTOMERS`, and `PRODUCTS`. Return a dict with exactly these keys:

`orders_total`, `known_customer`, `missing_customer`, `unknown_customer`, `known_product`, `missing_product`, `unknown_product`, `orphan_orders`, `customers_without_orders`.

- A missing key is null; an unknown key is non-null but absent from the dimension table.
- `orphan_orders` counts order rows with a missing or unknown product.
- `customers_without_orders` counts customers with no order rows.
- Use `validate="many_to_one"` and `indicator=True` for the joins.
- Do not mutate the inputs.

**Hint:** a left anti-join is a left merge followed by `_merge == "left_only"`.


In [ ]:

def audit_joins(orders, customers, products):
    

In [ ]:

before_orders = ORDERS.copy(deep=True)
before_customers = CUSTOMERS.copy(deep=True)
before_products = PRODUCTS.copy(deep=True)

audit = audit_joins(ORDERS, CUSTOMERS, PRODUCTS)

assert isinstance(audit, dict)
assert audit == {
    "orders_total": 1770,
    "known_customer": 1683,
    "missing_customer": 51,
    "unknown_customer": 36,
    "known_product": 1735,
    "missing_product": 18,
    "unknown_product": 17,
    "orphan_orders": 35,
    "customers_without_orders": 15,
}
pd.testing.assert_frame_equal(ORDERS, before_orders)
pd.testing.assert_frame_equal(CUSTOMERS, before_customers)
pd.testing.assert_frame_equal(PRODUCTS, before_products)
print("Task 3 passed")


<details>
<summary>Show reference solution</summary>

```python
def audit_joins(orders, customers, products):
    customer_join = orders.merge(
        customers[["customer_id"]],
        on="customer_id",
        how="left",
        validate="many_to_one",
        indicator=True,
    )
    product_join = orders.merge(
        products[["product_id"]].drop_duplicates(),
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator=True,
    )
    anti = customers[["customer_id"]].merge(
        orders[["customer_id"]].dropna().drop_duplicates(),
        on="customer_id",
        how="left",
        indicator=True,
    )
    return {
        "orders_total": len(orders),
        "known_customer": int(customer_join["_merge"].eq("both").sum()),
        "missing_customer": int(orders["customer_id"].isna().sum()),
        "unknown_customer": int(
            (
                ~orders["customer_id"].isna()
                & ~orders["customer_id"].isin(customers["customer_id"])
            ).sum()
        ),
        "known_product": int(product_join["_merge"].eq("both").sum()),
        "missing_product": int(orders["product_id"].isna().sum()),
        "unknown_product": int(
            (
                ~orders["product_id"].isna()
                & ~orders["product_id"].isin(products["product_id"])
            ).sum()
        ),
        "orphan_orders": int(product_join["_merge"].eq("left_only").sum()),
        "customers_without_orders": int(anti["_merge"].eq("left_only").sum()),
    }
```

</details>


## Task 4 - Daily report (9 minutes)

Use `BASE`. Write `daily_report(base)` returning a dict with:

- `observed`: daily revenue, indexed by normalized `order_date`, sorted;
- `full`: `observed` reindexed to every calendar day in the observed range, missing days filled with 0;
- `monthly`: month-end totals from `full` using `resample("ME")`;
- `rolling_7`: 7-day rolling mean of `full` with `min_periods=1`;
- `cumulative`: cumulative revenue of `full`.

The index of `full` must be a daily `DatetimeIndex`; the index of `monthly` must be month-end. Do not mutate `base`.

**Hint:** `pd.date_range(...)` plus `reindex(..., fill_value=0)` creates the complete calendar.


In [ ]:

def daily_report(base):
    ...


In [ ]:

before = BASE.copy(deep=True)
report = daily_report(BASE)

assert isinstance(report, dict)
assert set(report) == {"observed", "full", "monthly", "rolling_7", "cumulative"}

observed = report["observed"]
full = report["full"]
monthly = report["monthly"]
rolling = report["rolling_7"]
cumulative = report["cumulative"]

assert len(observed) == 788
assert len(full) == 1096
assert len(monthly) == 36
assert np.isclose(observed.sum(), 1299800.79)
assert np.isclose(full.sum(), 1299800.79)
assert np.isclose(monthly.sum(), 1299800.79)
assert str(full.index.freq) == "<Day>"
assert str(monthly.index.freq) == "<MonthEnd>"
assert monthly.index[0] == pd.Timestamp("2022-01-31")
assert monthly.index[-1] == pd.Timestamp("2024-12-31")
assert full.notna().all()
assert np.isclose(rolling.iloc[0], full.iloc[0])
assert np.isclose(rolling.iloc[-1], full.iloc[-7:].mean())
assert np.isclose(cumulative.iloc[-1], full.sum())
pd.testing.assert_frame_equal(BASE, before)
print("Task 4 passed")


<details>
<summary>Show reference solution</summary>

```python
def daily_report(base):
    observed = (
        base.groupby(base["order_date"].dt.normalize())["revenue"]
        .sum()
        .sort_index()
    )
    full = observed.reindex(
        pd.date_range(observed.index.min(), observed.index.max(), freq="D"),
        fill_value=0,
    )
    monthly = full.resample("ME").sum()
    return {
        "observed": observed,
        "full": full,
        "monthly": monthly,
        "rolling_7": full.rolling(7, min_periods=1).mean(),
        "cumulative": full.cumsum(),
    }
```

</details>


## Task 5 - Reshape report (9 minutes)

Use `BASE`. Write `reshape_report(base)` returning a dict with:

- `wide`: `pivot_table` with month periods as rows, canonical channels as columns, summed revenue as values, zeros for missing combinations, and `Total` margins;
- `long`: `wide` without the `Total` row/column, melted to columns `month`, `channel`, `revenue`, dropping missing revenue;
- `share`: `crosstab` of channel by status, row-normalized, with an `All` row margin, rounded to 3 decimals.

Do not mutate `base`.

**Hint:** `pivot_table(..., margins=True, margins_name="Total")`; `melt(id_vars="month", var_name="channel", value_name="revenue")`; `crosstab(..., normalize="index", margins=True, margins_name="All")`.


In [ ]:

def reshape_report(base):
    ...


In [ ]:

before = BASE.copy(deep=True)
report = reshape_report(BASE)

assert isinstance(report, dict)
assert set(report) == {"wide", "long", "share"}

wide = report["wide"]
long = report["long"]
share = report["share"]

assert wide.index.name == "month"
assert wide.columns.name == "channel"
assert wide.shape == (37, 4)
assert set(wide.columns) == {"mobile", "store", "web", "Total"}
assert np.isclose(wide.loc["Total", "Total"], 1299800.79)
assert wide.drop(index="Total", columns="Total").notna().all().all()

assert list(long.columns) == ["month", "channel", "revenue"]
assert len(long) == 108
assert long["revenue"].notna().all()
assert np.isclose(
    long["revenue"].sum(),
    wide.drop(index="Total", columns="Total").to_numpy().sum(),
)

assert share.index.name == "channel"
assert share.columns.name == "status"
assert set(share.index) == {"mobile", "store", "web", "All"}
assert {"completed", "cancelled", "pending", "returned"}.issubset(share.columns)
assert np.allclose(share.drop(index="All").sum(axis=1), 1.0, atol=0.002)
assert np.isclose(share.loc["All"].sum(), 1.0, atol=0.002)
pd.testing.assert_frame_equal(BASE, before)
print("Task 5 passed")


<details>
<summary>Show reference solution</summary>

```python
def reshape_report(base):
    wide = (
        base.assign(month=base["order_date"].dt.to_period("M"))
        .pivot_table(
            index="month",
            columns="channel",
            values="revenue",
            aggfunc="sum",
            fill_value=0,
            margins=True,
            margins_name="Total",
        )
    )
    long = (
        wide.drop(index="Total").drop(columns="Total").reset_index()
        .melt(id_vars="month", var_name="channel", value_name="revenue")
        .dropna(subset=["revenue"])
    )
    share = pd.crosstab(
        base["channel"],
        base["status"],
        normalize="index",
        margins=True,
        margins_name="All",
    ).round(3)
    return {"wide": wide, "long": long, "share": share}
```

</details>


## Task 6 - As-of price attachment (7 minutes)

Write `attach_price_asof(orders, prices)` using `CLEAN` and raw `PRICES`. Attach the most recent `list_price` known on or before each order date for the order's product.

- Keep only orders with a parsed `order_date` and a product present in prices.
- Parse `effective_date` with `format="mixed"`, `errors="coerce"`.
- Use `merge_asof` with `by="product_id"` and `direction="backward"`.
- Sort the result by `order_date`.
- Include `effective_date` and `list_price`; keep all original columns.
- Do not mutate the inputs.

**Hint:** both frames must be sorted by the as-of key; `by` partitions the match per product.


In [ ]:

def attach_price_asof(orders, prices):
    ...


In [ ]:

before_orders = CLEAN.copy(deep=True)
before_prices = PRICES.copy(deep=True)
out = attach_price_asof(CLEAN, PRICES)

eligible = CLEAN.dropna(subset=["order_date"])
eligible = eligible[eligible["product_id"].isin(PRICES["product_id"])]

assert len(out) == len(eligible) == 1615
assert "list_price" in out.columns
assert "effective_date" in out.columns
assert out["list_price"].notna().all()
assert out["order_date"].is_monotonic_increasing
assert (out["effective_date"] <= out["order_date"]).all()

sample = out.iloc[0]
prices_parsed = PRICES.assign(
    effective_date=pd.to_datetime(PRICES["effective_date"], format="mixed", errors="coerce")
)
expected = (
    prices_parsed.loc[
        (prices_parsed["product_id"] == sample["product_id"])
        & (prices_parsed["effective_date"] <= sample["order_date"])
    ]
    .sort_values("effective_date")["list_price"]
    .iloc[-1]
)
assert np.isclose(sample["list_price"], expected)
pd.testing.assert_frame_equal(CLEAN, before_orders)
pd.testing.assert_frame_equal(PRICES, before_prices)
print("Task 6 passed")


<details>
<summary>Show reference solution</summary>

```python
def attach_price_asof(orders, prices):
    prices = prices.assign(
        effective_date=pd.to_datetime(
            prices["effective_date"], format="mixed", errors="coerce"
        ),
        list_price=pd.to_numeric(prices["list_price"], errors="coerce"),
    )
    left = (
        orders.dropna(subset=["order_date"])
        .loc[lambda frame: frame["product_id"].isin(prices["product_id"])]
        .sort_values(["order_date", "product_id"])
    )
    right = prices.sort_values(["effective_date", "product_id"])
    return pd.merge_asof(
        left,
        right,
        left_on="order_date",
        right_on="effective_date",
        by="product_id",
        direction="backward",
        suffixes=("", "_price"),
    )
```

</details>


## Bonus A - Top customers by country (3 minutes)

Write `top_customers_by_country(base, n=3)` returning columns `country`, `customer_id`, `revenue`, `rank` with the top `n` customers per country by revenue. Sort by country and rank.


In [ ]:

def top_customers_by_country(base, n=3):
    ...


In [ ]:

top = top_customers_by_country(BASE, n=3)

assert list(top.columns) == ["country", "customer_id", "revenue", "rank"]
assert len(top) == 18
assert top.groupby("country").size().eq(3).all()
assert set(top["country"]) == set(BASE["country"])
assert top["rank"].between(1, 3).all()
print("Bonus A passed")


<details>
<summary>Show reference solution</summary>

```python
def top_customers_by_country(base, n=3):
    return (
        base.groupby(["country", "customer_id"])["revenue"]
        .sum()
        .reset_index()
        .assign(
            rank=lambda frame: frame.groupby("country")["revenue"].rank(
                method="first", ascending=False
            )
        )
        .loc[lambda frame: frame["rank"] <= n]
        .sort_values(["country", "rank"])
        .reset_index(drop=True)
    )
```

</details>


## Bonus B - Category share (2 minutes)

Write `category_share(base)` returning revenue and share by category, sorted by revenue descending.


In [ ]:

def category_share(base):
    ...


In [ ]:

shares = category_share(BASE)

assert list(shares.columns) == ["revenue", "share"]
assert len(shares) == 4
assert np.isclose(shares["share"].sum(), 1.0)
assert np.isclose(shares["revenue"].sum(), 1299800.79)
assert shares["revenue"].is_monotonic_decreasing
print("Bonus B passed")


<details>
<summary>Show reference solution</summary>

```python
def category_share(base):
    out = base.groupby("category")["revenue"].sum().to_frame("revenue")
    return (
        out.assign(share=out["revenue"] / out["revenue"].sum())
        .sort_values("revenue", ascending=False)
    )
```

</details>


## Stop the timer

### Post-mortem

Answer these in a new markdown cell or on paper:

1. Which task took the longest, and why?
2. Which self-check failed first?
3. Did any function mutate an input?
4. Which task used `.pipe` or a named helper chain?
5. Which pattern will you memorize before the interview?
6. Re-run this notebook in 2 days and compare.

### Target recall

- `.loc` is label-inclusive; `.iloc` is position-exclusive.
- Assume nothing: assert columns, index, dtypes, uniqueness, and totals.
- `agg` reduces; `transform` broadcasts; `filter` keeps whole groups.
- `merge(validate=..., indicator=True)` is how you make joins auditable.
- `pivot_table`, `melt`, and `crosstab` are the three reshaping tools to know cold.
- `reindex(pd.date_range(...), fill_value=0)` then `resample("ME")` is the time-series pattern.
- `merge_asof(..., by=..., direction="backward")` attaches the latest prior value.
- `.pipe` makes a pipeline readable when each step has a name.
